# Mimicry and DDN against the defences

Two more attacks measured against the defended models. **Part A** runs **mimicry** — graft benign features onto malware until the detector relents — against every model in the section: the plain DNN, both adversarially trained models, ICNN, PAD, DLA, DNN+, and KDE. **Part B** runs the **DDN** attack (two phases: optimise, then discretise) against AT-PGD and AT-MaxMA, including transfer attacks between them, and repeats mimicry in that notebook's own form.

**Where this fits:** Section 03 built the attacks; this section asks how the **defences** hold up against them — above all against the σ attack that grew out of `03-attacks/cw_sine_lineage.ipynb`. The same defences, measured in the final organised protocol, are `06-thesis-evaluation/`.

*Consolidated from `Mimicry.ipynb` and `DDN&mimicry.ipynb` (originally in `PGD_form_attacks`). Outputs cleared.*

## Part A — Mimicry against every model

### Step 1 — Setup

# Loading data

In [ ]:
import torch
import torch.nn as nn

# Assuming you are using a GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import gdown

download_links = [
                  'https://drive.google.com/uc?id=1dkoFDyJMuGewWX-LE5rb7zkRU6XkSa2z',
                  'https://drive.google.com/uc?id=1oHJBbqyKybmlFFRZ7-uftn77Vg0Hr_A5',
                  'https://drive.google.com/uc?id=1YwxR5CCNnG5gjeFBlBA8mEfDQM22mLj_',
                  'https://drive.google.com/uc?id=1XB5jdefjnGSxrb27SO-65JZpz5UcsNh2',
                  'https://drive.google.com/uc?id=1Ahw4prIhm322LU53Yhkju5ixd6q1KCJ6',
                  'https://drive.google.com/uc?id=18_YhP6qHHmu52CfANYD2LqxrOzq5ECxl',
                  'https://drive.google.com/uc?id=140wKHyRpKurSuvEOswKdyH3kKRPg8VW_',
                  'https://drive.google.com/uc?id=1pTokv8xKvZoUo8kejJc7N757iNfHmt2s',
                  'https://drive.google.com/uc?id=107sDEWVIJZJWC1SIWASEepdRXB3VWlCZ',
                  'https://drive.google.com/uc?id=1sFBZ3A8VQ-iDjmwASg4u5uHQQR8kVJy9',


]


output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import zipfile

# Define the path to the zip file and the extraction path
zip_file_path = '/content/datasets.zip'
extract_to_path = '/content/'

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load tensors
train_features = torch.load('train_features.pt', weights_only=True).float()
train_labels = torch.load('train_labels.pt', weights_only=True).long()
val_features = torch.load('val_features.pt', weights_only=True).float()
val_labels = torch.load('val_labels.pt', weights_only=True).long()
test_features = torch.load('test_features.pt', weights_only=True).float()
test_labels = torch.load('test_labels.pt', weights_only=True).long()

# Print data shapes for verification
print(f"Train Features Shape: {train_features.shape}")
print(f"Train Labels Shape: {train_labels.shape}")
print(f"Validation Features Shape: {val_features.shape}")
print(f"Validation Labels Shape: {val_labels.shape}")
print(f"Test Features Shape: {test_features.shape}")
print(f"Test Labels Shape: {test_labels.shape}")

# Create datasets
train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)

# Set batch size
batch_size = 512
# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Optional: Verify DataLoader sizes
print('----------------------------------------------')
print(f"Train Loader Batch Count: {len(train_loader)}")
print(f"Validation Loader Batch Count: {len(val_loader)}")
print(f"Test Loader Batch Count: {len(test_loader)}")

In [ ]:
# Function to count samples for each label
def count_labels(labels):
    count_0 = (labels == 0).sum().item()  # Count samples with label = 0
    count_1 = (labels == 1).sum().item()  # Count samples with label = 1
    return count_0, count_1

# Count labels for each dataset
train_count_0, train_count_1 = count_labels(train_labels)
val_count_0, val_count_1 = count_labels(val_labels)
test_count_0, test_count_1 = count_labels(test_labels)

# Calculate total counts across all datasets
total_count_0 = train_count_0 + val_count_0 + test_count_0
total_count_1 = train_count_1 + val_count_1 + test_count_1

# Print results
print(f"Training set             \t Label 0: {train_count_0}\t Label 1: {train_count_1}")
print(f"Validation set           \t Label 0: {val_count_0}\t Label 1: {val_count_1}")
print(f"Test set                 \t Label 0: {test_count_0}\t Label 1: {test_count_1}")
print(f"Total across all datasets\t Label 0: {total_count_0}\t Label 1: {total_count_1}")

In [ ]:
print(f"The avrage number of feature in Training set: {train_features.sum()/train_features.shape[0]:.2f}")
print(f"The avrage number of feature in Validation set: {val_features.sum()/val_features.shape[0]:.2f}")
print(f"The avrage number of feature in Test set: {test_features.sum()/test_features.shape[0]:.2f}")

In [ ]:
removal_array = torch.load('manipulation_x.pt', weights_only=True, map_location=device)
removal_array.sum()

In [ ]:
import torch

def final_rounding(oimgs, newimg, model, removal_array, is_detector_enabled=False, oblivion=False):
    """
    Apply final rounding adjustments to adversarial images based on the model's predictions.

    Parameters:
    - oimgs (Tensor): Original images tensor.
    - newimg (Tensor): Adversarially perturbed images tensor.
    - model (nn.Module): The model used to evaluate active samples.
    - removal_array (Tensor): Array indicating features that should not be modified.

    Returns:
    - final (Tensor): The adjusted adversarial images.
    """

    # Ensure tensors are on the correct device
    device = oimgs.device
    oimgs, perturbed, removal_array = oimgs.to(device), newimg.to(device), removal_array.to(device)

    # Create mask for non-fixed features as a boolean tensor
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).bool()

    # Set the model to evaluation mode
    model.eval()

    # Calculate differences and prepare significant change mask
    with torch.no_grad():

        if is_detector_enabled:
            if model.name == 'kde':
                logits, x_hidden = model.forward_f(oimgs)
                prob = model.forward_g(x_hidden, y_pred=0)
            else:
                logits, prob = model.forward(oimgs)
        else:
            logits = model.forward(oimgs)

        if is_detector_enabled and not oblivion:
            tau = model.tau[0].item()
            inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
        else:
            inactive_mask = (logits.argmax(dim=1) == 0)

        active_mask = ~inactive_mask

        # Calculate total change for each sample
        total_change = (perturbed - oimgs).abs()
        significant_mask = total_change > 0.1
        sorted_idx = total_change.argsort(dim=1, descending=True)

        # Define changes to be applied, ensuring correct type and device
        changes = torch.where(perturbed > oimgs, 1, torch.where(perturbed < oimgs, 0, oimgs)).float().to(device)

    # Clone the original images for final adjustments
    final = oimgs.clone()

    # Set a practical maximum for significant changes per sample
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 500)

    # Iterate through each significant change index
    for i in range(max_significant_changes):
        if not active_mask.any():
            break

        active_indices = torch.where(active_mask)[0]

        # Update the final image with changes based on non-fixed feature mask
        indices = sorted_idx[active_indices, i]
        final[active_indices, indices] = torch.where(
            non_fixed_features_mask[active_indices, indices],
            changes[active_indices, indices],
            final[active_indices, indices]
        )

        # Update active_mask based on model prediction after applying changes
        with torch.no_grad():

            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(final[active_indices])
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(final[active_indices])
            else:
                logits = model.forward(final[active_indices])

            if is_detector_enabled and not oblivion:
                inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
            else:
                inactive_mask = (logits.argmax(dim=1) == 0)

            active_mask[active_indices] = (~inactive_mask)

    return final

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, removal_array):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - removal_array: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `removal_array` is expanded to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")

    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def calculate_distances(adv_batch, mal_x_batch, mask, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_batch (torch.Tensor): Adversarial samples batch.
    - mal_x_batch (torch.Tensor): Original samples batch.
    - mask (torch.Tensor): Boolean mask for successful attacks.
    - norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    adv_samples = adv_batch[mask]
    original_samples = mal_x_batch[mask]

    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")


def plot_adv_accuracies(adv_rounded_list, labels, model, mal_x_list, is_detector_enabled=False, device='cpu', batch_size=512, norm='L2', interval=5):
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D', 'v']

    # Disable LaTeX rendering to avoid errors
    plt.rcParams['text.usetex'] = False

    plt.figure(figsize=(18, 12))
    model.eval()
    x_min, x_max = 0, 0

    for idx, (adv_rounded, mal_x, color, label) in tqdm(enumerate(zip(adv_rounded_list, mal_x_list, colors, labels)),
                                                  total=len(adv_rounded_list), desc="Processing Tensors"):
        adv_rounded = adv_rounded.to(device)
        num_samples = adv_rounded.size(0)
        all_distances = []

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            adv_batch = adv_rounded[start:end]
            mal_x_batch = mal_x[start:end]

            with torch.no_grad():
                if is_detector_enabled:
                    if model.name == 'kde':
                        logits, x_hidden = model.forward_f(adv_batch)
                        prob = model.forward_g(x_hidden, y_pred=0)
                    else:
                        logits, prob = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
                else:
                    logits = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0)

            if successful_attacks_mask.any():
                distances = calculate_distances(adv_batch, mal_x_batch, successful_attacks_mask, norm)
                all_distances.append(distances)

        if all_distances:
            total_distances = torch.cat(all_distances)
            sorted_distances, _ = torch.sort(total_distances)
            x_values = sorted_distances.cpu()

            x_min = min(x_min, int(x_values.min().item()))
            x_max = max(x_max, int(x_values.max().item()))

            # Calculate percentage values for accuracy on y-axis
            y_values = (1 - torch.arange(1, len(sorted_distances) + 1).float().cpu() / num_samples) * 100

            sns.lineplot(x=x_values, y=y_values, marker=markers[idx % len(markers)], color=color,
                         linestyle=line_styles[idx % len(line_styles)], linewidth=2, markersize=5, alpha=0.8, label=label)

    plt.xlabel(f'Distance ({norm} norm)', fontsize=14, labelpad=10)
    plt.ylabel('Model Accuracy (%)', fontsize=14, labelpad=10)
    #plt.title('Model Accuracy vs. Distance from Adversarial Examples', fontsize=16, pad=10)
    plt.title(f'Accuracy (%) of detector {model.name} vs. Distance under attack', fontsize=16, pad=10)
    plt.legend(title='Adversarial Sets', fontsize=10, title_fontsize=12, loc='upper right')

    # Set x-axis with integer values but fewer ticks
    plt.grid(True, which='major', linestyle='--', linewidth=0.5)
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', linewidth=0.5)

    # Y-axis ticks for percentage values, every 10% from 0% to 100%
    plt.yticks(ticks=range(0, 110, 10))

    # Set x-axis ticks at intervals, e.g., every 5 units
    plt.xticks(ticks=range(x_min, x_max + 1, interval))

    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from typing import Any, Tuple

def attack_mal_batches(
    test_loader: DataLoader,
    model: nn.Module,
    attack: Any,
    removal_array: torch.Tensor,
    device: torch.device,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    indicator_masking: bool = False,
    **kwargs
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Applies adversarial attacks on malicious samples within batches and returns
    the original malicious samples and the corresponding adversarial examples.

    Args:
        test_loader (DataLoader): DataLoader for test data.
        model (nn.Module): Model to be attacked.
        attack (Any): Adversarial attack function (e.g., sigma_zero_rounded).
        removal_array (torch.Tensor): Masking tensor indicating removable features.
        device (torch.device): Device for computation.
        is_detector_enabled (bool, optional): Enables/disables detector during attack.
        oblivion (bool, optional): Flag for attack configuration.
        indicator_masking (bool, optional): Enables indicator masking for suspecious samples (prob>tau).
        **kwargs: Additional parameters for the attack function (e.g., steps, lr).

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: Original malicious samples and adversarial examples.
    """
    model.eval()
    perturbed_batches = []
    perturbed_indicator_batches = []
    mal_x_batches = []

    # Iterate through each batch in the test_loader
    for x_test, y_test in test_loader:
        torch.cuda.empty_cache()

        # Move batch to device
        x_test, y_test = x_test.to(device), y_test.to(device)

        # Select malicious samples (label == 1)
        mal_x_batch = x_test[y_test.squeeze() == 1]

        if mal_x_batch.size(0) > 0:  # Process only if there are malicious samples
            # Generate adversarial examples for malicious samples
            perturbed_batch = attack(
                model=model,
                inputs=mal_x_batch,
                removal_array=removal_array,
                device=device,
                is_detector_enabled=is_detector_enabled,
                oblivion=oblivion,
                **kwargs
            )
            perturbed_batches.append(perturbed_batch)
            mal_x_batches.append(mal_x_batch)

            # Apply indicator masking if enabled and required
            if is_detector_enabled and not oblivion and indicator_masking:
                print(f"{'-'*50}")
                print('Oblivion Attack ')
                perturbed_indicator_batch = attack(
                    model=model,
                    inputs=mal_x_batch,
                    removal_array=removal_array,
                    device=device,
                    is_detector_enabled=is_detector_enabled,
                    oblivion=True,
                    **kwargs
                )
                perturbed_indicator_batches.append(perturbed_indicator_batch)

    # Combine all malicious samples and their perturbed counterparts
    combined_mals = torch.cat(mal_x_batches, dim=0)
    combined_advs = torch.cat(perturbed_batches, dim=0)

    # Initialize final adversarial examples based on attack conditions
    if is_detector_enabled and not oblivion and indicator_masking:
        combined_indicator_advs = torch.cat(perturbed_indicator_batches, dim=0)
        final_advs = combined_advs.clone()

        # Get logits and probability
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(final_advs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model(final_advs)

        tau = model.tau[0].item()
        success_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)

        # Update unsuccessful adversarial examples using the indicator batch
        final_advs = torch.where(success_mask.unsqueeze(-1), final_advs, combined_indicator_advs)
    else:
        final_advs = combined_advs

    return combined_mals, final_advs

In [ ]:
import torch
from torch import nn
from typing import List

def calculate_distance(adv_samples: torch.Tensor, original_samples: torch.Tensor, norm: str) -> torch.Tensor:
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Args:
        adv_samples (torch.Tensor): The adversarial samples.
        original_samples (torch.Tensor): The original samples.
        norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: Calculated distances between each adversarial and original sample.
    """
    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")

def sum_of_top_n_distances(adv_rounded: torch.Tensor, mal_x_batch: torch.Tensor, y_pred: torch.Tensor, n: int, norm: str = 'L1') -> torch.Tensor:
    """
    Calculate the sum of the top n smallest distances between adv_rounded and mal_x_batch
    for samples where y_pred == 0, based on the specified norm.

    Args:
        adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
        mal_x_batch (torch.Tensor): The original malicious examples tensor.
        y_pred (torch.Tensor): The prediction tensor.
        n (int): The number of top distances to sum.
        norm (str): The norm to use ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: The sum of the top n smallest distances.
    """
    distances = calculate_distance(adv_rounded[y_pred == 0], mal_x_batch[y_pred == 0], norm)
    top_n_distances = torch.topk(distances, n, largest=False).values
    return top_n_distances.sum()

def evaluate_adversarial_examples(
    mal_x: torch.Tensor,
    adv: torch.Tensor,
    model: nn.Module,
    removal_array: torch.Tensor,
    is_detector_enabled: bool = False,
    oblivion: bool = False,
    indicator_masking: bool = False,
    rounded: bool = False,
    batch_size: int = 1024,
    norm: str = 'L1'
) -> torch.Tensor:
    """
    Evaluates adversarial examples against a model, calculating various distance metrics
    and success rates for different norms, optionally in batches.

    Args:
        mal_x (torch.Tensor): Original malware input.
        adv (torch.Tensor): Adversarial examples generated from mal_x.
        model (nn.Module): The trained model to be evaluated.
        removal_array (torch.Tensor): Array used for final rounding.
        is_detector_enabled (bool, optional): Enables detector processing during evaluation.
        oblivion (bool, optional): Alters attack configuration if true.
        indicator_masking (bool, optional): Enables indicator masking for suspect samples.
        rounded (bool, optional): Applies final rounding on adversarial examples.
        batch_size (int, optional): Maximum samples per batch for processing.
        norm (str): Norm used for distance calculations ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: The final adversarial examples used in evaluation.
    """
    model.eval()
    torch.cuda.empty_cache()

    total_samples = mal_x.shape[0]
    num_successful_attacks = 0
    total_distances = 0
    y_pred_list = []
    mal_x_list = []
    final_adv_list = []
    all_distances: List[float] = []

    for i in range(0, total_samples, batch_size):
        mal_x_batch = mal_x[i:i + batch_size]
        adv_batch = adv[i:i + batch_size]

        # Apply rounding if specified
        final_adv_batch = (
            final_rounding(mal_x_batch, adv_batch, model, removal_array, is_detector_enabled, oblivion)
            if rounded else adv_batch.clone()
        )

        # Perform forward pass and determine successful attacks
        with torch.no_grad():
            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(final_adv_batch)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model(final_adv_batch)

                # Identify successful attacks
                successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())

                # Exclude samples for indicator masking
                if indicator_masking:
                    exclusion_mask = (logits.argmax(dim=1) == 0) & (prob > model.tau[0].item())
                    successful_attacks_mask = successful_attacks_mask[~exclusion_mask]
                    mal_x_batch = mal_x_batch[~exclusion_mask]
                    final_adv_batch = final_adv_batch[~exclusion_mask]

            else:
                logits = model(final_adv_batch)
                successful_attacks_mask = (logits.argmax(dim=1) == 0)

        final_adv_list.append(final_adv_batch)
        y_pred_batch = (~successful_attacks_mask).long()
        y_pred_list.append(y_pred_batch)
        mal_x_list.append(mal_x_batch)

        # Calculate distances for successful attacks
        num_successful_attacks += successful_attacks_mask.sum().item()
        if successful_attacks_mask.any():
            distances = calculate_distance(final_adv_batch[successful_attacks_mask], mal_x_batch[successful_attacks_mask], norm)
            total_distances += distances.sum().item()
            all_distances.extend(distances.cpu().tolist())

    # Aggregate predictions and adversarial examples
    y_pred = torch.cat(y_pred_list)
    final_adv = torch.cat(final_adv_list)
    final_mal_x = torch.cat(mal_x_list)

    # Calculate metrics
    num_considered_malwares = len(y_pred)
    accuracy = (y_pred == 1).float().mean().item() * 100
    mean_distance = total_distances / max(1, num_successful_attacks)
    median_distance = torch.median(torch.tensor(all_distances)).item() if all_distances else 0

    # Print results
    print(f"Evaluation on Adversarial Examples (Norm: {norm}, Rounded: {rounded})")
    print("-----------------------------------------------")
    print(f"Total malwares: {total_samples} and Considered malwares : {num_considered_malwares}")
    print(f"Successful attacks: {num_successful_attacks}/{num_considered_malwares}")
    print(f"Model accuracy: {accuracy:.2f}%")
    print(f"Mean distance (successful): {mean_distance:.2f}")
    print(f"Median distance (successful): {median_distance:.2f}")

    # Calculate top N distances
    top_n_values = [500, 1000, 2000, 2500, 3000]
    for top_n in top_n_values:
        if num_successful_attacks >= top_n:
            mean_top_n_distance = sum_of_top_n_distances(final_adv, final_mal_x, y_pred, top_n, norm=norm) / top_n
            print(f"Mean distance (TOP {top_n} successful): {mean_top_n_distance:.2f}")

    return final_adv, final_mal_x

# Models and attacks

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels

class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dropout = 0.6, alpha = 0.2, smooth = False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu


    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences)
        gt_labels = torch.cat(gt_labels, dim=0)
        return confidences, gt_labels


    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        """
        # evaluation
        confidence, y_true = self.inference(test_data_producer)
        y_pred = confidence.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
        print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

        if np.any([np.all(y_true == i) for i in range(self.n_classes)]):
            print("class absent.")
            return

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")


    def load(self, model_save_path):
        """
        load model parameters from disk
        """
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading rFGSM
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class AdvMalwareDetectorICNN(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='ICNN', ):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.md_nn_model = md_nn_model
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]


        if hasattr(self.md_nn_model, 'smooth'):
            if not self.md_nn_model.smooth:  # non-smooth, exchange it
                for name, child in self.md_nn_model.named_children():
                    if isinstance(child, nn.ReLU):
                        self.md_nn_model._modules['relu'] = nn.SELU()
        else:
            for name, child in self.md_nn_model.named_children():
                if isinstance(child, nn.ReLU):
                    self.md_nn_model._modules['relu'] = nn.SELU()
        self.md_nn_model = self.md_nn_model.to(self.device)

        # input convex neural network
        self.non_neg_dense_layers = []
        if len(self.dense_hidden_units) < 1:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                                       self.dense_hidden_units[i + 1],
                                                       bias=False))
        self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[-1], 1, bias=False))
        # registration
        for idx_i, dense_layer in enumerate(self.non_neg_dense_layers):
            self.add_module('non_neg_layer_{}'.format(idx_i), dense_layer)

        self.dense_layers = []
        self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        for i in range(len(self.dense_hidden_units[1:])):
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[i]))
        self.dense_layers.append(nn.Linear(self.input_size, 1))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('layer_{}'.format(idx_i), dense_layer)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward_f(self, x):
        return self.md_nn_model(x)

    def forward_g(self, x):
        prev_x = None
        for i, dense_layer in enumerate(self.dense_layers):
            x_add = []
            x1 = dense_layer(x)
            x_add.append(x1)
            if prev_x is not None:
                x2 = self.non_neg_dense_layers[i - 1](prev_x)
                x_add.append(x2)
            prev_x = torch.sum(torch.stack(x_add, dim=0), dim=0)
            if i < len(self.dense_layers):
                prev_x = F.selu(prev_x)
        return prev_x.reshape(-1)

    def forward(self, x):
        return self.forward_f(x), self.forward_g(x)

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)

        rtn_value = (y_pred == 0) & indicator_flag

        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

        return rtn_value

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(torch.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels


    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits_f, logits_g = self.forward(x)
        return torch.softmax(logits_f, dim=-1).detach().cpu().numpy(), logits_g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                x_logits = self.forward_g(x_val)
                probabilities.append(x_logits)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def reset_threshold(self):
        self.tau[0] = 0.


    def load(self, model_save_path):
        # load model
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading PAD
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels



class AMalwareDetectionDLA(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        assert len(self.dense_hidden_units) >= 1, "Expected at least one hidden layer."
        self.alarm_nn_model = TorchAlarm(input_size=sum(self.md_nn_model.dense_hidden_units))

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)

    def forward(self, x):
        hidden_activations = []
        for dense_layer in self.md_nn_model.dense_layers[:-1]:
            x = self.md_nn_model.activation_func(dense_layer(x))
            hidden_activations.append(x)
        logits = self.md_nn_model.dense_layers[-1](x)
        hidden_activations = torch.cat(hidden_activations, dim=-1)
        x_prob = self.alarm_nn_model(hidden_activations).reshape(-1)
        return logits, x_prob

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                print("class absent.")
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)
        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_logits = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1))
                x_prob.append(x_logits)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))


class TorchAlarm(torch.nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.layers = torch.nn.ModuleList([
            torch.nn.Linear(input_size, 112),
            torch.nn.ReLU(),
            torch.nn.Linear(112, 100),
            torch.nn.ReLU(),
            torch.nn.Linear(100, 300),
            torch.nn.ReLU(),
            torch.nn.Linear(300, 200),
            torch.nn.ReLU(),
            torch.nn.Linear(200, 77),
            torch.nn.ReLU(),
            torch.nn.Linear(77, 1),
        ])

    def __call__(self, x, training=False):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels







class AMalwareDetectionDNNPlus(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth

        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   self.n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        self.amd_nn_plus = MalwareDetectionDNN(self.input_size,
                                               self.n_classes + 1,
                                               self.device,
                                               name)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward(self, x):
        logits = self.amd_nn_plus(x)
        logits -= torch.amax(logits, dim=-1, keepdim=True).detach()  # increasing the stability, which might be helpful
        return logits, torch.softmax(logits, dim=-1)[:, -1]

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                print("class absent.")
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)
        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_cent = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1)[:, :2])
                x_prob.append(x_cent)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, g = self.forward(x)
        x_cent = torch.softmax(logits, dim=-1).detach().cpu().numpy()[:, :2]

        return x_cent, g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_cent = self.forward(x_val)
                probabilities.append(x_cent)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

In [ ]:
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class KernelDensityEstimation(nn.Module, DetectorTemplate):
    """
    kernel density estimation upon the penultimate layer

    parameters
    -------------
    @param model, torch.nn.Module, an instantiation of model object
    @param bandwidth, float, variance of Gaussian density function
    @param n_classes, integer, number of classes
    @param ratio, float [0,1], ratio for computing the threshold
    """

    def __init__(self, model, n_centers=1000, bandwidth=20., n_classes=2, ratio=0.95, name='KDE', device='cpu'):
        super(KernelDensityEstimation, self).__init__()
        assert isinstance(model, torch.nn.Module)
        self.model = model
        self.device = device
        self.name = name
        self.n_centers = n_centers
        self.bandwidth = bandwidth
        self.n_classes = n_classes
        self.ratio = ratio
        self.gaussian_means = None

        self.tau = nn.Parameter(torch.zeros([self.n_classes, ], device=self.device), requires_grad=False)

    def forward(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        x_prob = self.forward_g(x, logits.argmax(1).detach())
        return logits, x_prob

    def forward_f(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        return logits, x

    def forward_g(self, x_hidden, y_pred):
        """
        1 / |X_| * \sum(exp(||x - x_||_2^2/sigma^2))

        parameters
        -----------
        @param x_hidden, torch.tensor, node hidden representation
        @param y_pred, torch.tensor, prediction
        """
        size = x_hidden.size()[0]
        dist = [torch.sum(torch.square(means.unsqueeze(dim=0) - x_hidden.unsqueeze(dim=1)), dim=-1) for means in
                self.gaussian_means]
        kd = torch.stack([torch.mean(torch.exp(-d / self.bandwidth ** 2), dim=-1) for d in dist], dim=1)
        # return p(x|y=y_pred)
        return -1 * kd[torch.arange(size), y_pred]

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for density estimation
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        self.eval()
        probabilities = []
        gt_labels = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                logits, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
                gt_labels.append(y_val)
            prob = torch.cat(probabilities, dim=0)
            gt_labels = torch.cat(gt_labels)
            for i in range(self.n_classes):
                prob_x_y = prob[gt_labels == i]
                s, _ = torch.sort(prob_x_y)
                self.tau[i] = s[int((s.shape[0] - 1) * ratio)]

    def predict(self, test_data_producer, indicator_masking=True):
        # evaluation on detector & indicator
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob, y_pred).cpu().numpy()
        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.

        print('The indicator is turning on...')
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print(f"Number of considered samples is {len(y_true)}%")
        print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
        print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")


        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print(f"False Negative Rate (FNR) is {fnr:.5f}%, False Positive Rate (FPR) is {fpr:.5f}%, F1 score is {f1:.5f}%")


    def eval(self):
        self.model.eval()

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = x.to(device), y.to(device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(F.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)

        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau[y_pred]

    def indicator(self, x_prob, y_pred=None):
        assert y_pred is not None
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.get_tau_sample_wise(y_pred)).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.get_tau_sample_wise(y_pred)
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")
        # res = probability.reshape(-1, 1).repeat_interleave(2, dim=1) >= self.tau
        # return res[torch.arange(res.size()[0]), y_pred]


    def load(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.gaussian_means = ckpt['gaussian_means']
        self.tau = ckpt['tau']
        self.model.load_state_dict(ckpt['base_model'])

### Step 2 — Loading the models (compact: one load + check pattern per model)

## Load DNN

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

In [ ]:
model_dnn.double().to(device)
model_dnn.predict(test_loader)

In [ ]:
model_dnn.float()

## Load rFGSM

In [ ]:
model_rFGSM = MalwareDetectionDNN(10000,2,device=device, name='rFGSM')
model_rFGSM.load2('/content/model_at_pgd.pth')

In [ ]:
model_rFGSM.double().to(device)
model_rFGSM.predict(test_loader)

In [ ]:
model_rFGSM.float()

## Load MaxMA

In [ ]:
model_MaxMA = MalwareDetectionDNN(10000,2,device=device, name='MaxMA')
model_MaxMA.load2('/content/model_at_ma.pth')

In [ ]:
model_MaxMA.double().to(device)
model_MaxMA.predict(test_loader)

In [ ]:
model_MaxMA.float()

## Loadin ICNN

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_icnn = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,name='ICNN')
model_icnn.load('/content/amd_icnn.pth')
model_icnn.to(device).float()

In [ ]:
model_icnn.double().to(device)
model_icnn.predict(test_loader, indicator_masking=True)

In [ ]:
model_icnn.to(device).float()

In [ ]:
model_icnn.tau

In [ ]:
model_icnn.tau[0].item()

## Loadin PAD

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_pad = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,name='PAD')
model_pad.load2('/content/amd_pad_ma.pth')
model_pad.to(device).float()

In [ ]:
model_pad.double()
model_pad.predict(test_loader, indicator_masking=True)

In [ ]:
model_pad.to(device).float()

In [ ]:
model_pad.tau

In [ ]:
model_pad.tau[0].item()

## Check DLA

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_dla = AMalwareDetectionDLA(model_dnn,input_size=10000,n_classes=2,device=device,name='DLA')
model_dla.load('/content/amd_dla.pth')
model_dla.to(device).float()

In [ ]:
model_dla.double()
model_dla.predict(test_loader, indicator_masking=True)

In [ ]:
model_dla.float()

In [ ]:
model_dla.tau

In [ ]:
model_dla.tau[0].item()

## Check DNN_Plus

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_dnn_plus = AMalwareDetectionDNNPlus(model_dnn,input_size=10000,n_classes=2,device=device,name='DNNPlus')
model_dnn_plus.load('/content/amd_dnn_plus.pth')
model_dnn_plus.to(device).float()

In [ ]:
model_dnn_plus.double()
model_dnn_plus.predict(test_loader, indicator_masking=True)

In [ ]:
model_dnn_plus.float()

In [ ]:
model_dnn_plus.tau

In [ ]:
model_dnn_plus.tau[0].item()

### Step 3 — The Mimicry function and the batch processor

# Mimicry

In [ ]:
import torch

def Mimicry(model, inputs, input_bens, removal_array, is_detector_enabled=False, oblivion=False, device=device, verbose=True):
    """
    Performs mimicry-based adversarial attack to find benign samples similar to malicious ones
    and modifies malicious samples to deceive the model.

    Args:
        model: The PyTorch model under attack.
        inputs: Malicious inputs (batch of examples).
        input_bens: Benign inputs for mimicry.
        removal_array: Mask specifying which features to modify.
        is_detector_enabled: Whether a detector is enabled in the model.
        oblivion: Specific condition for detector logic.
        device: Device to perform computations (e.g., 'cpu' or 'cuda').

    Returns:
        final_adv: Tensor of adversarial examples.
    """
    torch.cuda.empty_cache()

    # Move necessary tensors to the specified device
    model = model.to(device)
    model.eval()
    inputs = inputs.to(device)
    input_bens = input_bens.to(device)
    removal_array = removal_array.to(device)
    fixed_features_mask = torch.bitwise_and(~removal_array.unsqueeze(0), inputs.to(torch.uint8))

    # Initialize storage tensors and counters
    final_adv = inputs.clone()
    total_samples = 0
    valid_sample_count = 0
    valid_distances_sum = 0.0

    # Iterate through the malicious inputs
    for idx, mal_sample in enumerate(inputs):

        # Combine malicious sample with benign samples
        ben_x_batch_new = torch.cat((mal_sample.unsqueeze(0) , input_bens), dim=0)
        inputs_combined = torch.bitwise_or(fixed_features_mask[idx].unsqueeze(0), ben_x_batch_new.to(torch.uint8)).float()
        # Pass the combined inputs through the model
        with torch.no_grad():
            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(inputs_combined)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(inputs_combined)
            else:
                logits = model.forward(inputs_combined)

            success_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item() if is_detector_enabled and not oblivion else True)

        # Calculate L1 distances and filter successful inputs
        distances = (mal_sample != inputs_combined).sum(dim=1)
        valid_distances = distances[success_mask]
        valid_inputs = inputs_combined[success_mask]

        # Update counters and tensors with valid results
        total_samples += 1
        if valid_distances.numel() > 0:
            min_distance, min_index = torch.min(valid_distances, dim=0)
            final_adv[idx] = valid_inputs[min_index]
            valid_sample_count += 1
            valid_distances_sum += min_distance.item()

        # Logging progress
        if verbose and ((idx + 1) % 512 == 0):
            current_mean_distance = valid_distances_sum / valid_sample_count if valid_sample_count > 0 else 0.0
            print(f"Processed {idx + 1:4}/{inputs.size(0):4} samples | "
                  f"Success: {valid_sample_count:4}/{total_samples:4} | "
                  f"Avg. Dist: {current_mean_distance:3.2f}")

    return final_adv

In [ ]:
import torch

def process_batches(loader, device):
    """
    Processes batches from a data loader and splits them into malicious and benign samples.

    Args:
        loader: DataLoader for the dataset (train, validation, or test).
        device: Device to transfer tensors to ('cpu' or 'cuda').

    Returns:
        A tuple containing:
            - mal_x (Tensor): Malicious samples.
            - ben_x (Tensor): Benign samples.
            - mal_y (Tensor): Labels for malicious samples.
            - ben_y (Tensor): Labels for benign samples.
    """
    ben_x_list = []

    for x_batch, y_batch in loader:
        # Transfer data to the specified device
        x_batch = x_batch.to(torch.float32).to(device)
        y_batch = y_batch.to(device)

        # Split into malicious and benign batches
        ben_mask = y_batch.squeeze() == 0

        ben_x_list.append(x_batch[ben_mask])


    # Concatenate all batches
    ben_x = torch.cat(ben_x_list, dim=0) if ben_x_list else torch.empty(0, *x_batch.shape[1:], device=device)

    return ben_x

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Process training, validation, and test loaders
ben_x_train = process_batches(train_loader, device)
ben_x_val = process_batches(val_loader, device)
ben_x_test= process_batches(test_loader, device)

# Combine benign samples from all sets
ben_samples_all = torch.cat((ben_x_train, ben_x_val, ben_x_test), dim=0)
ben_samples_part = torch.cat((ben_x_val, ben_x_test), dim=0)
ben_samples_test = ben_x_test

# Logging
print('------------------------------------------------------')
print(f"Combined All Benign Samples: {ben_samples_all.shape}")
print(f"Combined (Validation & Test) Benign Samples: {ben_samples_part.shape}")
print(f"Only Test Benign Samples: {ben_samples_test.shape}")

### Step 4 — Mimicry vs DNN, AT-rFGSM, AT-MaxMA

## model_dnn

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_dnn,Mimicry,removal_array,device,is_detector_enabled=False,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_all,verbose=True)

result, malwares = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_dnn, removal_array=removal_array,
                                              is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result,result2]
mal_x_list = [malwares,malwares2]
labels = ['all_data','test_data']
plot_adv_accuracies(adv_rounded_list, labels, model_dnn, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L0', interval=10)

## model_rFGSM

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_rFGSM,Mimicry,removal_array,device,is_detector_enabled=False,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_all,verbose=True)

result, malwares = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_rFGSM, removal_array=removal_array,
                                              is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result,result2]
mal_x_list = [malwares,malwares2]
labels = ['all_data','test_data']
plot_adv_accuracies(adv_rounded_list, labels, model_rFGSM, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L0', interval=10)

## model_MaxMA

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_MaxMA,Mimicry,removal_array,device,is_detector_enabled=False,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_all,verbose=True)

result, malwares = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_MaxMA, removal_array=removal_array,
                                              is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result,result2]
mal_x_list = [malwares,malwares2]
labels = ['all_data','test_data']
plot_adv_accuracies(adv_rounded_list, labels, model_MaxMA, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L0', interval=10)

### Step 5 — Mimicry vs the detectors (ICNN, PAD, DLA, DNN+)

## model_icnn

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_icnn,Mimicry,removal_array,device,is_detector_enabled=True,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_all,verbose=True)

result_detect, malwares_detect = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_icnn, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_icnn,Mimicry,removal_array,device,is_detector_enabled=True,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_test,verbose=True)

result_detect2, malwares_detect2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_icnn, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result_detect,result_detect2,result_indic,result_indic2,result_obliv,result_obliv2]
mal_x_list = [malwares_detect,malwares_detect2,malwares_indic,malwares_indic2,malwares_obliv,malwares_obliv2]
labels = ['all_data_detector','test_data_detector','all_data_indicator','test_data_indicator','all_data_oblivion','test_data_oblivion']
plot_adv_accuracies(adv_rounded_list, labels, model_icnn, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=10)

## model_pad

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_pad,Mimicry,removal_array,device,is_detector_enabled=True,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_all,verbose=True)

result_detect, malwares_detect = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_pad, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_pad,Mimicry,removal_array,device,is_detector_enabled=True,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_test,verbose=True)

result_detect2, malwares_detect2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_pad, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result_detect,result_detect2,result_indic,result_indic2,result_obliv,result_obliv2]
mal_x_list = [malwares_detect,malwares_detect2,malwares_indic,malwares_indic2,malwares_obliv,malwares_obliv2]
labels = ['all_data_detector','test_data_detector','all_data_indicator','test_data_indicator','all_data_oblivion','test_data_oblivion']
plot_adv_accuracies(adv_rounded_list, labels, model_pad, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=10)

## model_dla

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_dla,Mimicry,removal_array,device,is_detector_enabled=True,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_all,verbose=True)

result_detect, malwares_detect = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_dla, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_dla,Mimicry,removal_array,device,is_detector_enabled=True,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_test,verbose=True)

result_detect2, malwares_detect2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_dla, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result_detect,result_detect2,result_indic,result_indic2,result_obliv,result_obliv2]
mal_x_list = [malwares_detect,malwares_detect2,malwares_indic,malwares_indic2,malwares_obliv,malwares_obliv2]
labels = ['all_data_detector','test_data_detector','all_data_indicator','test_data_indicator','all_data_oblivion','test_data_oblivion']
plot_adv_accuracies(adv_rounded_list, labels, model_dla, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=10)

## model_dnn_plus

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_dnn_plus,Mimicry,removal_array,device,is_detector_enabled=True,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_all,verbose=True)

result_detect, malwares_detect = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_dnn_plus, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
combined_mals, combined_advs = attack_mal_batches(test_loader,model_dnn_plus,Mimicry,removal_array,device,is_detector_enabled=True,
                                                        oblivion=False,indicator_masking =False,input_bens=ben_samples_test,verbose=True)

result_detect2, malwares_detect2 = evaluate_adversarial_examples(mal_x=combined_mals, adv=combined_advs, model=model_dnn_plus, removal_array=removal_array,
                                              is_detector_enabled=True, oblivion=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
adv_rounded_list = [result_detect,result_detect2,result_indic,result_indic2,result_obliv,result_obliv2]
mal_x_list = [malwares_detect,malwares_detect2,malwares_indic,malwares_indic2,malwares_obliv,malwares_obliv2]
labels = ['all_data_detector','test_data_detector','all_data_indicator','test_data_indicator','all_data_oblivion','test_data_oblivion']
plot_adv_accuracies(adv_rounded_list, labels, model_dnn_plus, mal_x_list, is_detector_enabled=True, device=device, batch_size=batch_size, norm='L0', interval=10)

### Step 6 — The KDE check that closes the notebook

# New Section

## Check KDE

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_kde = KernelDensityEstimation(model_dnn,n_centers=1000,bandwidth=16.,n_classes=2,ratio=0.95,device=device,name='KDE')
model_kde.load('/content/amd_kde.pth')
model_kde.to(device).float()

In [ ]:
model_kde.predict(test_loader, indicator_masking=True)

In [ ]:
model_kde.predict(test_loader, indicator_masking=False)

In [ ]:
model_kde.tau

In [ ]:
logits, x_prob = model_kde.forward(mal_x_batch[:10])
print(logits, x_prob)

In [ ]:
(x_prob <model_kde.tau[1]).sum()

In [ ]:
model_kde((mal_x_batch[:5]+0.01))

In [ ]:
model_kde((mal_x_batch[:5]+0.05))

## Part B — DDN and mimicry against the adversarially trained models

### Step 7 — Setup: data, harness, and the two AT models

# Loading data

In [ ]:
import torch

# Assuming you are using a GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import gdown

download_links = [
                  'https://drive.google.com/uc?id=1dkoFDyJMuGewWX-LE5rb7zkRU6XkSa2z',
                  'https://drive.google.com/uc?id=1oHJBbqyKybmlFFRZ7-uftn77Vg0Hr_A5',
                  'https://drive.google.com/uc?id=1YwxR5CCNnG5gjeFBlBA8mEfDQM22mLj_',
                  'https://drive.google.com/uc?id=1XB5jdefjnGSxrb27SO-65JZpz5UcsNh2',
                  'https://drive.google.com/uc?id=1Ahw4prIhm322LU53Yhkju5ixd6q1KCJ6',
                  'https://drive.google.com/uc?id=18_YhP6qHHmu52CfANYD2LqxrOzq5ECxl',
                  'https://drive.google.com/uc?id=140wKHyRpKurSuvEOswKdyH3kKRPg8VW_',
                  'https://drive.google.com/uc?id=1pTokv8xKvZoUo8kejJc7N757iNfHmt2s',
                  'https://drive.google.com/uc?id=107sDEWVIJZJWC1SIWASEepdRXB3VWlCZ',


]


output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load tensors
train_features = torch.load('train_features.pt', weights_only=True).float()
train_labels = torch.load('train_labels.pt', weights_only=True).long()
val_features = torch.load('val_features.pt', weights_only=True).float()
val_labels = torch.load('val_labels.pt', weights_only=True).long()
test_features = torch.load('test_features.pt', weights_only=True).float()
test_labels = torch.load('test_labels.pt', weights_only=True).long()

# Print data shapes for verification
print(f"Train Features Shape: {train_features.shape}")
print(f"Train Labels Shape: {train_labels.shape}")
print(f"Validation Features Shape: {val_features.shape}")
print(f"Validation Labels Shape: {val_labels.shape}")
print(f"Test Features Shape: {test_features.shape}")
print(f"Test Labels Shape: {test_labels.shape}")

# Create datasets
train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)

# Set batch size
batch_size = 20000

# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Optional: Verify DataLoader sizes
print('----------------------------------------------')
print(f"Train Loader Batch Count: {len(train_loader)}")
print(f"Validation Loader Batch Count: {len(val_loader)}")
print(f"Test Loader Batch Count: {len(test_loader)}")

In [ ]:
removal_array = torch.load('manipulation_x.pt', weights_only=True)
removal_array.sum()

In [ ]:
import torch

def calculate_accuracy(model, test_loader, device='cpu'):
    """
    Calculate the accuracy of the model on the test dataset.

    Parameters:
    ----------
    model : nn.Module
        The trained model.
    test_loader : DataLoader
        The DataLoader containing the test dataset.
    device : str
        Device to run the model on ('cpu' or 'cuda').

    Returns:
    -------
    float
        The accuracy of the model on the test dataset.
    """
    # Set the model to evaluation mode
    model.eval()

    # Track the number of correct predictions
    correct_predictions = 0
    total_samples = 0

    # Disable gradient computation
    with torch.no_grad():
        for inputs, labels in test_loader:
            # Move data to the selected device
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)

            # Get the predicted class (index of the highest logit value)
            _, predicted = torch.max(outputs, 1)

            # Count correct predictions
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    # Calculate accuracy
    accuracy = correct_predictions / total_samples
    return accuracy

In [ ]:
def sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, n):
    """
    Calculate the sum of the top n smallest differences between adv_rounded and mal_x_batch
    for samples where y_pred == 0.

    Parameters:
    adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
    mal_x_batch (torch.Tensor): The original malicious examples tensor.
    y_pred (torch.Tensor): The prediction tensor.
    n (int): The number of top differences to sum.

    Returns:
    torch.Tensor: The sum of the top n smallest differences.
    """
    # Step 1: Calculate the total differences
    total_difference = torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0]).sum(1)

    # Step 2: Sort the differences in ascending order
    sorted_differences, _ = torch.sort(total_difference)

    # Step 3: Select the top n least differences
    top_n_least_differences = sorted_differences[:n]

    # Step 4: Sum these n differences
    sum_top_n_least_differences = top_n_least_differences.sum()

    return sum_top_n_least_differences

In [ ]:
def final_rounding(oimgs, newimg, model, removal_array):

    # Ensure all tensors are on the correct device
    oimgs, perturbed = oimgs.to(device), newimg.to(device)
    removal_array = removal_array.to(device)

    # Expand removal array to match the batch size
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8))

    # Set the model to evaluation mode
    model.eval()

    # Calculate predictions and significant changes efficiently using GPU
    with torch.no_grad():
        predictions = model(oimgs).argmax(dim=1).to(device)
        active_mask = predictions == 1  # Efficiently create active mask on GPU
        total_change = (perturbed - oimgs).abs().to(device)  # Calculate absolute difference on GPU
        significant_mask = total_change > 0.1  # Determine significant changes on GPU
        sorted_idx = total_change.argsort(dim=1, descending=True)  # Sort on GPU
        #changes = torch.where(perturbed > oimgs, torch.tensor(1), torch.where(perturbed < oimgs, torch.tensor(0), 1 - oimgs )).float().to(device)
        changes = torch.where(perturbed > oimgs, torch.tensor(1), torch.where(perturbed < oimgs, torch.tensor(0), oimgs)).float().to(device)

    # Copy original images to apply final rounding
    final = oimgs.clone().to(device)

    # Iterate up to the maximum number of significant changes
    # practicaly we dont have perturbs more than 120 !!
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 200)

    for i in range(max_significant_changes):
        # Safeguard against empty active mask
        if not active_mask.any():
            break

        # Get the indices of active samples
        active_indices = torch.where(active_mask)[0].to(device)

        # Get the relevant indices for current change
        indices = sorted_idx[active_indices, i]

        # since max_significant_changes is relatively large for some samples, possible that zero changes which belong to fixed_features get updated
        final[active_indices, indices] = torch.where(non_fixed_features_mask[active_indices, indices],
                                                     changes[active_indices, indices], final[active_indices, indices])

        # Update the active_mask based on the model's prediction
        with torch.no_grad():
            active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

In [ ]:
import torch

def final_rounding(oimgs, newimg, model, removal_array):
    """
    Apply final rounding adjustments to adversarial images based on the model's predictions.

    Parameters:
    - oimgs (Tensor): Original images tensor.
    - newimg (Tensor): Adversarially perturbed images tensor.
    - model (nn.Module): The model used to evaluate active samples.
    - removal_array (Tensor): Array indicating features that should not be modified.

    Returns:
    - final (Tensor): The adjusted adversarial images.
    """

    # Ensure tensors are on the correct device
    device = oimgs.device
    oimgs, perturbed, removal_array = oimgs.to(device), newimg.to(device), removal_array.to(device)

    # Create mask for non-fixed features as a boolean tensor
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).bool()

    # Set the model to evaluation mode
    model.eval()

    # Calculate differences and prepare significant change mask
    with torch.no_grad():
        predictions = model(oimgs).argmax(dim=1)
        active_mask = predictions == 1
        total_change = (perturbed - oimgs).abs()
        significant_mask = total_change > 0.1
        sorted_idx = total_change.argsort(dim=1, descending=True)

        # Define changes to be applied, ensuring correct type and device
        changes = torch.where(perturbed > oimgs, 1, torch.where(perturbed < oimgs, 0, oimgs)).float().to(device)

    # Clone the original images for final adjustments
    final = oimgs.clone()

    # Set a practical maximum for significant changes per sample
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 200)

    # Iterate through each significant change index
    for i in range(max_significant_changes):
        if not active_mask.any():
            break

        active_indices = torch.where(active_mask)[0]

        # Update the final image with changes based on non-fixed feature mask
        indices = sorted_idx[active_indices, i]
        final[active_indices, indices] = torch.where(
            non_fixed_features_mask[active_indices, indices],
            changes[active_indices, indices],
            final[active_indices, indices]
        )

        # Update active_mask based on model prediction after applying changes
        with torch.no_grad():
            active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

In [ ]:
def evaluate_adversarial_examples(mal_x_batch, mal_y_batch, adv, model, removal_array, rounded=True):
    """
    Evaluate the model's performance on adversarial examples, optionally applying rounding.

    Parameters:
    - mal_x_batch (Tensor): Original malware input batch.
    - mal_y_batch (Tensor): Ground truth labels for malware batch.
    - adv (Tensor): Adversarial examples generated from mal_x_batch.
    - model (nn.Module): The trained model to be evaluated.
    - removal_array (Tensor): Array used for final rounding.
    - rounded (bool, optional): Whether to apply rounding on adversarial examples. Default is True.

    Returns:
    - final_adv (Tensor): The (rounded) adversarial examples used in evaluation.
    """

    # Set the model to evaluation mode
    model.eval()

    # Apply rounding if specified
    final_adv = final_rounding(mal_x_batch, adv, model, removal_array) if rounded else adv.clone()

    # Model predictions on adversarial examples
    with torch.no_grad():
        outputs = model(final_adv)
        y_pred = outputs.argmax(dim=1)

    # Evaluation metrics
    num_malwares = len(mal_y_batch)
    successful_attacks_mask = (y_pred == 0)
    num_successful_attacks = successful_attacks_mask.sum().item()

    # L1 norm differences for various evaluations
    def mean_l1_diff(adv_samples, orig_samples, top_n=None):
        if top_n is not None:
            return sum_of_top_n_differences(adv_samples, orig_samples, y_pred, top_n) / top_n
        return (torch.abs(adv_samples - orig_samples)).sum().item() / max(1, num_successful_attacks)

    # Only calculate for successful adversarial examples
    total_difference = mean_l1_diff(final_adv[successful_attacks_mask], mal_x_batch[successful_attacks_mask])
    total_difference_TOP100 = mean_l1_diff(final_adv, mal_x_batch, top_n=100)
    total_difference_TOP500 = mean_l1_diff(final_adv, mal_x_batch, top_n=500)
    total_difference_TOP1000 = mean_l1_diff(final_adv, mal_x_batch, top_n=1000)

    # Accuracy
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print evaluation results
    print("Evaluation Results on Rounded Adversarial Examples")
    print("-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Mean difference (successful adv. examples vs. original): {total_difference:.2f}")
    print(f"Mean difference (successful adv. examples vs. original) - TOP 100  : {total_difference_TOP100:.2f}")
    print(f"Mean difference (successful adv. examples vs. original) - TOP 500  : {total_difference_TOP500:.2f}")
    print(f"Mean difference (successful adv. examples vs. original) - TOP 1000 : {total_difference_TOP1000:.2f}")
    print(f"Model accuracy on malware under attack: {accuracy:.2f}%")

    return final_adv

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def plot_adv_distances(adv_rounded_list, labels, model, mal_x_batch2, device):
    """
    Plots the cumulative distance between adversarial tensors and a reference batch (mal_x_batch2).

    Parameters:
    - adv_rounded_list: List of adversarial tensors to be evaluated.
    - labels: List of labels for the adversarial tensors (for the plot legend).
    - model: The model used for predicting adversarial examples.
    - mal_x_batch2: The reference batch to compare adversarial tensors against.
    - device: Device to run the computations on (e.g., 'cuda' or 'cpu').
    - model_AT_rFGSM: A variant of the model to evaluate adversarial tensors.
    """

    # Set a style using Seaborn and select a color palette automatically
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))  # Automatically select colors

    # Create the plot
    plt.figure(figsize=(20, 12))

    model.eval()

    # Iterate through each adv_rounded tensor and plot their distances
    for adv_rounded, color, label in tqdm(zip(adv_rounded_list, colors, labels),
                                          total=len(adv_rounded_list), desc="Processing Tensors"):
        # Pass the adv_rounded tensor through the model to get predictions
        adv_rounded = adv_rounded.to(device)  # Move tensor to the correct device
        outputs = model(adv_rounded)
        y_pred = outputs.argmax(dim=1)

        # Calculate the differences between adv_rounded and mal_x_batch2 for correctly classified samples
        mal_x_batch2 = mal_x_batch2.to(device)
        total_difference = torch.sum((adv_rounded[y_pred == 0] - mal_x_batch2[y_pred == 0]) ** 2, dim=1)

        # Sort the differences in ascending order
        sorted_differences, _ = torch.sort(total_difference)

        # Generate the x-axis (distance values) and y-axis (cumulative counts)
        x_values = sorted_differences.cpu()  # Move tensor back to CPU for plotting
        y_values = torch.arange(1, len(sorted_differences) + 1).cpu()  # Create a tensor for the y-axis values

        # Plot the line for the current adv_rounded tensor
        sns.lineplot(x=x_values, y=y_values, marker='o', color=color, linewidth=2, markersize=5, label=label)

    # Add labels and title with a bit more formatting
    plt.xlabel('Distance', fontsize=16, labelpad=10)
    plt.ylabel('Number of Valid Samples (Cumulative)', fontsize=16, labelpad=10)
    plt.title('Cumulative Count of Valid Samples vs. Distance', fontsize=18, pad=15)

    # Add legend for clarity
    plt.legend(title='Adversarial Tensors', fontsize=12, title_fontsize=14, loc='upper left')

    # Improve the look of the grid and remove spines for a cleaner look
    plt.grid(True, linestyle='--', linewidth=0.5)
    sns.despine()

    # Show the plot with tighter layout for better visualization
    plt.tight_layout()
    plt.show()

# Loading models and attacks

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', **kwargs):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.device = device
        self.name = name

        self.parse_args(**kwargs)

        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu


    def parse_args(self,
                   dense_hidden_units=None,
                   dropout=0.6,
                   alpha_=0.2,
                   smooth=False,
                   **kwargs
                   ):
        if dense_hidden_units is None:
            self.dense_hidden_units = [200, 200]
        elif isinstance(dense_hidden_units, list):
            self.dense_hidden_units = dense_hidden_units
        else:
            raise TypeError("Expect a list of hidden units.")

        self.dropout = dropout
        self.alpha_ = alpha_
        self.smooth = smooth
        self.proc_number = kwargs['proc_number']


    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits


    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

In [ ]:
for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      print(mal_x_batch.shape)
      print(mal_y_batch.shape)
      break

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device,proc_number = 10)
model_dnn.load_state_dict(torch.load('/content/model_dnn.pth', map_location=torch.device(device)))
model_dnn.to(device)

# Calculate accuracy on the test set
test_accuracy = calculate_accuracy(model_dnn, test_loader, device)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
model_at_pgd = MalwareDetectionDNN(10000,2,device=device,proc_number = 10)

# Load the saved dictionary from the file
checkpoint = torch.load('/content/model_at_pgd.pth', map_location=torch.device('cpu'))

# Extract the model's state_dict from the checkpoint
model_state_dict = checkpoint['model']

# Load the extracted state_dict into the model
model_at_pgd.load_state_dict(model_state_dict)

# Move model to device
model_at_pgd.to(device)

# Calculate accuracy on the test set
test_accuracy = calculate_accuracy(model_at_pgd, test_loader, device)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
model_at_ma = MalwareDetectionDNN(10000,2,device=device,proc_number = 10)
# Load the saved dictionary from the file
checkpoint = torch.load('/content/model_at_ma.pth', map_location=torch.device('cpu'))

# Extract the model's state_dict from the checkpoint
model_state_dict = checkpoint['model']

# Load the extracted state_dict into the model
model_at_ma.load_state_dict(model_state_dict)

# Move model to device
model_at_ma.to(device)

# Calculate accuracy on the test set
test_accuracy = calculate_accuracy(model_at_ma, test_loader, device)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

### Step 8 — DDN phase 1 and phase 2

# DDN

In [ ]:
# optimizer mode

import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

def DNN_phase1(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 0.1,
    initial_gamma=0.01,
    grad_norm=torch.inf,
    verbose: bool = False,
    device = 'cpu'
):
    model.eval()
    model.to(device)

    batch_size = inputs.shape[0]
    # Create mask for non-fixed features as a boolean tensor
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(inputs.shape[0], -1), 1 - inputs.to(torch.uint8)).bool()
    def grad_hook(grad):
        return grad * non_fixed_features_mask

    # Initialize delta (perturbation) and optimizer
    delta = torch.zeros_like(inputs, requires_grad=True, device=device)
    optimizer = torch.optim.SGD([delta], lr=lr)
    #delta.register_hook(grad_hook)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=0.01)

    # Variables to track best solutions
    success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    best_l2 = torch.full((batch_size,), inputs.shape[1], device=device)
    best_delta = delta.clone()

    # for unsuccessfull attack samples
    best_delta_unsuccess = delta.clone()
    best_loss = torch.full((batch_size,), float('inf'), device=device)

    norm = torch.zeros((batch_size,1), device=device)
    gamma = torch.full((batch_size,1), initial_gamma, device=device)

    for step in range(steps):
        # Generate adversarial examples
        adv_inputs = inputs + delta

        # Calculate losses
        logits = model(adv_inputs)
        ben_score = logits[:, 0]
        mal_score = logits[:, 1]

        loss = mal_score - ben_score

        with torch.no_grad():
          #L2
          l2dist = delta.square().sum(dim=1)

          # Track successful adversarial examples
          predicted_labels = logits.argmax(1)
          success_mask_inloop = predicted_labels == 0
          success_samples  = torch.logical_or(success_mask_inloop, success_samples)

          improved_solution = (l2dist < best_l2) & success_mask_inloop
          best_l2 = torch.where(improved_solution, l2dist, best_l2)
          best_delta = torch.where(improved_solution.unsqueeze(-1), delta, best_delta)

          improved_solution_unsucces = (loss < best_loss)
          best_loss = torch.where(improved_solution_unsucces, loss, best_loss)
          best_delta_unsuccess = torch.where(improved_solution_unsucces.unsqueeze(-1), delta, best_delta_unsuccess)


        # Calculate adversarial loss
        adv_loss = loss.mean()

        if verbose and (step + 1) % 100 == 0:
            print(f"Iteration {step + 1:5}:\t Current LR= {optimizer.param_groups[0]['lr']:.2f}\t Current success = {success_mask_inloop.sum().item()}\t success = {success_samples.sum().item()}\t Dist = {best_l2[success_samples].mean().item():.2f}\t Loss = {loss[success_samples].mean().item():.2f}")

        # Update delta
        optimizer.zero_grad()
        adv_loss.backward()


        gradients = delta.grad.data
        grad4removal = (gradients > 0) * (adv_inputs > 0.) * non_fixed_features_mask * gradients
        grad4insertion = (gradients < 0) *(adv_inputs < 1.)* gradients
        gradients = grad4removal + grad4insertion

        grad_norms = gradients.norm(p=grad_norm, dim=1, keepdim=True).clamp_(min=1e-12)
        gradients /= grad_norms

        # Apply opposite direction for unsuccessful samples
        delta.grad.data = torch.where(success_mask_inloop.unsqueeze(-1),
                                      -gradients,
                                      gradients)


        '''
        # Normalize gradient
        grad_norms = delta.grad.data.norm(p=grad_norm, dim=1, keepdim=True).clamp_(min=1e-12)
        delta.grad.data.div_(grad_norms)

        #delta_grad_flattened = delta.grad.data.flatten(1)
        #grad_norms = delta_grad_flattened.norm(p=grad_norm, dim=1, keepdim=True).clamp_(min=1e-12)
        #delta.grad.data = (delta_grad_flattened / grad_norms).view(delta.grad.shape)

        # Apply opposite direction for unsuccessful samples
        delta.grad.data = torch.where(success_mask_inloop.unsqueeze(-1),
                                      -delta.grad.data,
                                      delta.grad.data)
        '''

        optimizer.step()
        scheduler.step()

        #shrink eps for success samples
        gamma = torch.where(success_mask_inloop.unsqueeze(-1),
                            gamma*0.9,
                            gamma)

        delta_norm = torch.linalg.norm(delta, dim=-1, keepdim=True).clamp_(min=1e-12)
        update_mask = (delta_norm > norm)
        norm = torch.where(update_mask,
                          norm+gamma,
                          delta_norm)

        delta.data.mul_((norm / delta_norm))

        # Clamp delta to maintain valid pixel range
        delta.data.add_(inputs.data).clamp_(min=0, max=1).sub_(inputs.data)

    with torch.no_grad():
        logits = model(inputs + best_delta)
        final_success = logits.argmax(1)==0
        final_delta = torch.where(final_success.unsqueeze(-1), best_delta.detach(), best_delta_unsuccess.detach())

    # Return the adversarial examples with the best delta found
    return inputs + final_delta

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR
import math
import numpy as np


# Define the learning rate schedules (same as earlier)
def lr_linear(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start + t * (lr_end - lr_start) / T

def lr_exponential(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_start * (lr_end / lr_start) ** (t / T)

def lr_cosine(t, T, lr_start=1e-3, lr_end=0.9):
    return lr_end + (lr_start - lr_end) * 0.5 * (1 + torch.cos(torch.tensor(np.pi * t / T)))

def lr_average(t, T, lr_start=1e-3, lr_end=0.9):
    return 0.5 * (lr_linear(t, T, lr_start, lr_end) + lr_exponential(t, T, lr_start, lr_end))

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))


def DDN_phase1(x, y, model, removal_array, max_iterations, min_lr, max_lr, initial_eps=0.01, device='device'):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)

    # Initialize starting point
    x_next = x.clone().to(device)
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1).to(device)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = x.clone().detach()
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)

    # for unsuccessfull attack samples
    o_best_unsuccessfull_attack = x.clone().detach()
    o_best_loss = torch.full((batch_size,), float('inf'), device=device)

    #norm = 0.
    norm = torch.zeros((batch_size,1), device=device)
    eps = torch.full((batch_size,1), initial_eps, device=device)

    for t in range(max_iterations):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)

        # Calculate losses
        logits = model(x_var)
        ben_score = logits[:, 0]
        mal_score = logits[:, 1]

        loss = mal_score - ben_score

        with torch.no_grad():

            l2dist = torch.sum((x - x_next) ** 2, dim=1)

            predicted_labels = logits.argmax(1)
            success_mask_inloop = predicted_labels == 0
            success_mask  = torch.logical_or(success_mask_inloop, success_mask)

            update_mask = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask, l2dist, o_bestl2)
            o_bestattack = torch.where(update_mask.unsqueeze(-1), x_var, o_bestattack)

            update_mask2 = (loss < o_best_loss)
            o_best_loss = torch.where(update_mask2, loss, o_best_loss)
            o_best_unsuccessfull_attack = torch.where(update_mask2.unsqueeze(-1), x_var, o_best_unsuccessfull_attack)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients
        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients

        gradients = grad4removal + grad4insertion

        l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True).clamp_(min=1e-12)
        perturbation = (gradients / l2norm )



        # Update x_next
        step_length = lr_cosine(t, max_iterations, min_lr, max_lr)
        #delta = (x_next + perturbation * step_length) - x
        delta = torch.where(success_mask_inloop.unsqueeze(-1),
                          (x_next - perturbation * step_length) - x,
                          (x_next + perturbation * step_length) - x)

        #shrink eps for success samples
        eps = torch.where(success_mask_inloop.unsqueeze(-1),
                          eps*0.9,
                          eps)

        delta_norm = torch.linalg.norm(delta, dim=-1, keepdim=True).clamp_(min=1e-12)
        update_mask = (delta_norm > norm)
        norm = torch.where(update_mask,
                          norm+eps,
                          delta_norm)

        x_next = torch.clamp(x + norm * (delta/delta_norm), min=0., max=1.)


        '''
        norm += 0.01
        delta_norm = torch.linalg.norm(delta, dim=-1, keepdim=True)
        update_mask = (delta_norm > norm)

        #print(update_mask.shape)
        x_next = torch.where(update_mask,
                             torch.clamp(x + norm * (delta/(delta_norm + 1e-20)), min=0., max=1.),
                             torch.clamp(x + delta, min=0., max=1.))
        '''

        if t == 0 or (t + 1) % 1000 == 0:
          #print('norm : ', norm)
          #print('delta_norm : ', delta_norm)
          print(f"Iteration {t + 1:5}:\t Current LR = {step_length:.4f}\t success = {success_mask.sum().item()}\t Dist = {o_bestl2[success_mask].mean().item():.2f}\t Loss = {loss[success_mask].mean().item():.2f}")

    logits = model(o_bestattack)
    final_success = (logits.argmax(1) == 0)
    final_output = torch.where(final_success.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR
import math
import numpy as np


def DDN_phase2(x, x0, model, removal_array, max_iterations, min_lr, max_lr, initial_eps=0.01, change=0.05, device='cpu'):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)

    # Initialize starting point
    x_next = x.clone().to(device)
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1).to(device)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestattack = x0.clone().detach()
    o_bestDist = torch.full((batch_size,), 1e10, device=device)
    #norm = 0.
    norm = torch.sum((x - x0) ** 2, dim=1, keepdim=True)
    eps = torch.full((batch_size,1), initial_eps, device=device)

    for t in range(max_iterations):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)

        # Calculate losses
        logits = model(x_var)
        ben_score = logits[:, 0]
        mal_score = logits[:, 1]

        loss = mal_score - ben_score

        with torch.no_grad():
            success_mask_inloop_not_rounded = (logits.argmax(1) == 0)

            x_next_rounded = final_rounding(x0, x_next, model, removal_array)
            dist = (torch.abs(x0 - x_next_rounded)).sum(1)

            logits = model(x_next_rounded)
            success_mask_inloop = (logits.argmax(1) == 0)
            success_mask  = torch.logical_or(success_mask_inloop, success_mask)

            update_mask2 = (dist < o_bestDist) & success_mask_inloop
            o_bestDist = torch.where(update_mask2, dist, o_bestDist)
            o_bestattack = torch.where(update_mask2.unsqueeze(-1), x_next_rounded, o_bestattack)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)

        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients
        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        gradients = grad4removal + grad4insertion

        l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True).clamp_(min=1e-12)
        perturbation = (gradients / l2norm)

        # Update x_next
        #step_length = cyclical_lr(t, step_size_up, step_size_down, min_lr, max_lr)
        step_length = lr_linear(t, max_iterations, min_lr, max_lr)
        #delta = (x_next + perturbation * step_length) - x
        delta = torch.where(success_mask_inloop_not_rounded.unsqueeze(-1),
                          (x_next - perturbation * step_length) - x,
                          (x_next + perturbation * step_length) - x)

        #shrink eps for success samples
        eps = torch.where(success_mask_inloop.unsqueeze(-1),
                          eps*(1-change),
                          eps*(1+change))

        delta_norm = torch.linalg.norm(delta, dim=-1, keepdim=True).clamp_(min=1e-12)
        update_mask = (delta_norm > norm)
        norm = torch.where(update_mask,
                          norm+eps,
                          delta_norm)

        x_next = torch.clamp(x + norm * (delta/delta_norm), min=0., max=1.)


        '''
        norm += 0.01
        delta_norm = torch.linalg.norm(delta, dim=-1, keepdim=True)
        update_mask = (delta_norm > norm)

        #print(update_mask.shape)
        x_next = torch.where(update_mask,
                             torch.clamp(x + norm * (delta/(delta_norm + 1e-20)), min=0., max=1.),
                             torch.clamp(x + delta, min=0., max=1.))
        '''

        if t == 0 or (t + 1) % 500 == 0:
          #print('norm : ', norm)
          #print('delta_norm : ', delta_norm)
          print(f"Iteration {t + 1:5}:\t Current LR = {step_length:.4f}\t success = {success_mask.sum().item()}\t Dist = {o_bestDist[success_mask].mean().item():.2f}\t Loss = {loss[success_mask].mean().item():.2f}")

    return o_bestattack

### Step 9 — DDN vs AT-PGD, then AT-MaxMA (with transfer)

## model_at_pgd

In [ ]:
adv = DDN_phase1(mal_x_batch, mal_y_batch, model_at_pgd, removal_array, max_iterations=5000, min_lr=0.05, max_lr=0.05, initial_eps=0.01, device=device)

In [ ]:
for r in [False,True]:
  results = evaluate_adversarial_examples(mal_x_batch, mal_y_batch, adv, model_at_pgd, removal_array, rounded=r)
  print()

In [ ]:
adv2 = DDN_phase2(adv, mal_x_batch, model_at_pgd, removal_array, max_iterations=2000, min_lr=0.5, max_lr=5., initial_eps=0.5, device=device)

In [ ]:
for r in [False,True]:
  results2 = evaluate_adversarial_examples(mal_x_batch, mal_y_batch, adv2, model_at_pgd, removal_array, rounded=r)
  print()

In [ ]:
adv_list = [adv,results,results2]  # Replace with your tensors
labels = ['DDN_phase1','DDN_phase1_rounded','DDN_phase2_rounded']  # Labels for the legend

plot_adv_distances(adv_list, labels, model_at_pgd, mal_x_batch, device)

## model_at_ma

In [ ]:
adv = DDN_phase1(mal_x_batch, mal_y_batch, model_at_ma, removal_array, max_iterations=5000, min_lr=0.05, max_lr=0.05, initial_eps=0.01, device=device)

In [ ]:
for r in [False,True]:
  results = evaluate_adversarial_examples(mal_x_batch, mal_y_batch, adv, model_at_ma, removal_array, rounded=r)
  print()

In [ ]:
adv2 = DDN_phase2(adv, mal_x_batch, model_at_ma, removal_array, max_iterations=3000, min_lr=0.5, max_lr=5., initial_eps=0.5, device=device)

In [ ]:
for r in [False,True]:
  results2 = evaluate_adversarial_examples(mal_x_batch, mal_y_batch, adv2, model_at_ma, removal_array, rounded=r)
  print()

In [ ]:
adv_list = [adv,results,results2]  # Replace with your tensors
labels = ['DDN_phase1','DDN_phase1_rounded','DDN_phase2_rounded']  # Labels for the legend

plot_adv_distances(adv_list, labels, model_at_ma, mal_x_batch, device)

In [ ]:
# TRANSFER ATTACK
adv_list = [adv,results,results2]  # Replace with your tensors
labels = ['DDN_phase1','DDN_phase1_rounded','DDN_phase2_rounded']  # Labels for the legend

plot_adv_distances(adv_list, labels, model_at_pgd, mal_x_batch, device)

### Step 10 — Mimicry in this notebook's form, vs both models

# Mimicry

In [ ]:
def Mimicry(model, removal_array, ben_x_batch, mal_x_batch):
    # Move the model and static tensors to the GPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    removal_array = removal_array.to(device)

    # Expand removal array to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1).to(torch.uint8)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Prepare tensors for storing results
    final_output_tensor1 = mal_x_batch.clone().to(device)
    final_output_tensor2 = torch.bitwise_and(
        1 - expanded_removal_array, mal_x_batch.to(torch.uint8)
    ).float().to(device)

    # Initialize counters and storage for valid distances
    total_samples = 0
    valid_sample_count = 0
    valid_distances_sum = 0.0
    valid_distances_list = []

    # Iterate through mal_x_batch with index tracking to store results
    for idx, x in enumerate(mal_x_batch):
        mal_sample = x.unsqueeze(0).to(device)
        ben_x_batch = ben_x_batch.to(device)

        non_removal_array_updated = torch.bitwise_and(
            1 - expanded_removal_array[0], mal_sample.to(torch.uint8)
        )

        # Create a batch of inputs by combining with benign samples
        ben_x_batch_new = torch.cat((mal_sample, ben_x_batch), dim=0)
        inputs = torch.bitwise_or(non_removal_array_updated, ben_x_batch_new.to(torch.uint8)).float()

        # Pass all inputs through the model in a batch
        with torch.no_grad():
            outputs = model(inputs)

        # Calculate L1 distances between the malicious input and the batch
        distances = torch.abs(mal_sample - inputs).sum(dim=1)

        # Filter out samples where the model predicts class 0 (mask == True)
        mask = outputs.argmax(dim=1) == 0
        valid_distances = distances[mask]
        valid_inputs = inputs[mask]

        # Increment the total sample counter
        total_samples += 1

        # Check for valid samples and update tensors
        if valid_distances.numel() > 0:
            min_distance, min_index = torch.min(valid_distances, dim=0)
            candidate_input = valid_inputs[min_index]

            # Store the valid candidate input into the final tensors
            final_output_tensor1[idx] = candidate_input
            final_output_tensor2[idx] = candidate_input

            # Update the valid sample count and distances
            valid_sample_count += 1
            valid_distances_sum += min_distance.item()
            valid_distances_list.append(min_distance.item())

        # Print status every 100 samples or at the start
        if idx == 0 or (idx + 1) % 1000 == 0:
            current_mean_distance = valid_distances_sum / valid_sample_count if valid_sample_count > 0 else 0
            print(f"samples {idx + 1:5}:\t Current success = {valid_sample_count:5} \t l1dist = {current_mean_distance:.2f}")

    # Final print statement for the last batch
    current_mean_distance = valid_distances_sum / valid_sample_count if valid_sample_count > 0 else 0
    print(f"samples {idx + 1:5}:\t Current success = {valid_sample_count:5} \t l1dist = {current_mean_distance:.2f}")

    # Calculate the percentage of samples with valid distances and mean distance
    valid_percentage = (1 - valid_sample_count / total_samples) * 100 if total_samples > 0 else 0
    mean_distance = valid_distances_sum / valid_sample_count if valid_sample_count > 0 else 0

    # Sort valid distances and calculate means for specific sets
    valid_distances_list.sort()
    first_100_mean = sum(valid_distances_list[:100]) / min(100, len(valid_distances_list)) if valid_distances_list else 0
    first_500_mean = sum(valid_distances_list[:500]) / min(500, len(valid_distances_list)) if valid_distances_list else 0
    first_1000_mean = sum(valid_distances_list[:1000]) / min(1000, len(valid_distances_list)) if valid_distances_list else 0

    # Print the final results
    print('-----------------------------------------------------------------------')
    print('Evaluation Results : ')
    print(f"Number of malware samples: {total_samples}")
    print(f"Number of successful attacks: {valid_sample_count}")
    print(f"Mean difference (successful adv. examples vs. original): {mean_distance:.2f}")
    print(f"Mean difference (successful adv. examples vs. original) - TOP 100  : {first_100_mean}")
    print(f"Mean difference (successful adv. examples vs. original) - TOP 500  : {first_500_mean}")
    print(f"Mean difference (successful adv. examples vs. original) - TOP 1000 : {first_1000_mean}")
    print(f"Model accuracy on malware under attack: {valid_percentage:.2f}%")

    return final_output_tensor1, final_output_tensor2, valid_distances_list

In [ ]:
# Extract batches from the validation set
for x_test, y_test in val_loader:
    x_test = x_test.to(torch.float32).to(device)
    y_test = y_test.to(device)

    # Split into malicious and benign batches based on labels
    mal_x_batch1 = x_test[y_test.squeeze() == 1]
    ben_x_batch1 = x_test[y_test.squeeze() == 0]

    mal_y_batch1 = y_test[y_test.squeeze() == 1]
    ben_y_batch1 = y_test[y_test.squeeze() == 0]

    print(f'Malicious batch (validation): {mal_x_batch1.shape}')
    print(f'Benign batch (validation): {ben_x_batch1.shape}')
    break

# Extract batches from the test set
for x_test, y_test in test_loader:
    x_test = x_test.to(torch.float32).to(device)
    y_test = y_test.to(device)

    # Split into malicious and benign batches based on labels
    mal_x_batch2 = x_test[y_test.squeeze() == 1]
    ben_x_batch2 = x_test[y_test.squeeze() == 0]

    mal_y_batch2 = y_test[y_test.squeeze() == 1]
    ben_y_batch2 = y_test[y_test.squeeze() == 0]

    print(f'Malicious batch (test): {mal_x_batch2.shape}')
    print(f'Benign batch (test): {ben_x_batch2.shape}')
    break

# Extract batches from the training set
for x_test, y_test in train_loader:
    x_test = x_test.to(torch.float32).to(device)
    y_test = y_test.to(device)

    # Split into malicious and benign batches based on labels
    mal_x_batch0 = x_test[y_test.squeeze() == 1]
    ben_x_batch0 = x_test[y_test.squeeze() == 0]

    mal_y_batch0 = y_test[y_test.squeeze() == 1]
    ben_y_batch0 = y_test[y_test.squeeze() == 0]

    print(f'Malicious batch (training): {mal_x_batch0.shape}')
    print(f'Benign batch (training): {ben_x_batch0.shape}')
    break

# Concatenate the benign batches from validation and test sets
ben_samples_all = torch.cat((ben_x_batch0, ben_x_batch1, ben_x_batch2), dim=0)
ben_samples_part = torch.cat((ben_x_batch1, ben_x_batch2), dim=0)
print('------------------------------------------------------')
print(f'Combined All benign samples: {ben_samples_all.shape}')
print(f'Combined (valid & test) benign samples: {ben_samples_part.shape}')

## model_at_pgd

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# all samples
final_output_tensor1, final_output_tensor2, valid_distances_list = Mimicry(model_at_pgd, removal_array, ben_samples_all, mal_x_batch2)

In [ ]:
# TRANSFER ATTACK
results = evaluate_adversarial_examples(mal_x_batch2, mal_y_batch2, final_output_tensor1, model_at_ma, removal_array, rounded=False)

In [ ]:
# part samples
final_output_tensor1, final_output_tensor2, valid_distances_list = Mimicry(model_at_pgd, removal_array, ben_samples_part, mal_x_batch2)

## model_at_ma

In [ ]:
# all samples
final_output_tensor1, final_output_tensor2, valid_distances_list = Mimicry(model_at_ma, removal_array, ben_samples_all, mal_x_batch2)

In [ ]:
# TRANSFER ATTACK
results = evaluate_adversarial_examples(mal_x_batch2, mal_y_batch2, final_output_tensor1, model_at_pgd, removal_array, rounded=False)

In [ ]:
# part samples
final_output_tensor1, final_output_tensor2, valid_distances_list = Mimicry(model_at_ma, removal_array, ben_samples_part, mal_x_batch2)